# RAG_time

Part 6's advisor got real information two ways: calling a live tool (`imt_taf_list`) or stuffing an entire researched briefing straight into its instructions (Program 14's `briefing_text`, built once for *every* TAF, whether the student ever asks about them or not). That works at ten TAF. It stops working once the source material is a 218-page book, a whole wiki, or a folder of PDFs -- Part 3's context window is finite, and most of that text would be irrelevant to any single question anyway.

**RAG** (Retrieval-Augmented Generation) is the fix: cut the source material into chunks and embed them once, in advance, embed the question the same way, and use cosine similarity to pull out only the few chunks actually relevant to it -- then hand *those* to the model, instead of everything.

Before building the actual retrieval system (that's Part 8), this part is about the piece RAG depends on entirely: turning a sentence into a vector that actually means something. Three ways to do it, compared head to head -- a homemade version, a real API, and a local model built for the job -- and, at the end, what's actually happening inside the model to make it possible.

## From token embeddings to sentence embeddings

Until now, embeddings lived at the token level: Part 3 showed a token is usually just a *piece* of a word, not the whole thing, and Part 4 gave each one its own vector. RAG needs something coarser -- a single vector for an entire sentence, so it can be compared to a whole question. A simple, homemade way to get there with any model: run the text through it, and average ("mean-pool") the resulting per-token vectors into a single one.

<br>
<img src="images/sentence_embedding_academic.jpg" width="550" alt="From Tokens to a Sentence Diagram" style="display: block; margin: 15px auto;">
<br>

### A quick word on PyTorch and Hugging Face

Two libraries have been doing quiet work since Part 3, without ever being properly introduced -- worth pausing on now that Program 1.1 imports `torch` directly for the first time, rather than just reading tensors a model handed back.

**PyTorch** (`import torch`) is an open-source library for numerical computing and machine learning, originally built by Facebook/Meta AI, and today the dominant framework for training and running deep learning models -- including the LLMs this whole course is built on. Two ideas make it what it is:

* **Tensors**, its core data structure -- essentially a NumPy array that can run on a GPU. Every `.mean()`, `.squeeze()`, and `@` (matrix multiply) used in this part is a tensor operation.
* **Autograd**, automatic differentiation: PyTorch tracks every operation performed on a tensor so it can compute gradients automatically when *training* a model. We're only doing inference here, never training, which is exactly why `sentence_embedding` wraps its work in `torch.no_grad()` below -- without it, PyTorch would keep tracking gradients for a training step that's never going to happen, for nothing.

**Hugging Face's `transformers`** is a separate library, built on top of PyTorch (or TensorFlow, or JAX -- your choice), that packages thousands of pretrained models behind one consistent interface. `AutoTokenizer.from_pretrained(name)` and `AutoModel.from_pretrained(name)`, used since Part 3, work the same way whatever model name you give them -- "Auto" means the library inspects the name and picks the right underlying tokenizer/model class for you.

In [ ]:
# Program 1.1: a homemade sentence embedding, by mean-pooling token vectors

import torch                     # tensors + autograd, see above
import torch.nn.functional as F  # tensor operations that aren't methods on the tensor itself, like cosine_similarity below
from transformers import AutoTokenizer, AutoModel

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoModel (not AutoModelForCausalLM, Part 3's choice) gives direct access to hidden
# states, without the extra layer that predicts next-token logits -- we don't need that here.
base_model = AutoModel.from_pretrained(model_name)
base_model.eval()  # inference mode: turns off training-only behaviour (like dropout) we don't want here

def sentence_embedding(text):
    """A simple sentence embedding: the average of all its tokens' final hidden states."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        hidden_states = base_model(**inputs).last_hidden_state  # shape: (1, num_tokens, embedding_dim)
    return hidden_states.mean(dim=1).squeeze(0)  # average over the tokens -> a single vector

sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

The two sentences that mean roughly the same thing (the cat/feline ones) score noticeably higher than either has with the unrelated sentence about the stock market -- even though they don't share a single word. That's the whole point of a sentence embedding: it captures meaning, not vocabulary overlap.

## Does this hold across languages?

Within English, the trick works cleanly. But a sentence embedding is only as good as what the underlying model actually learned -- and `SmolLM2-135M` was trained overwhelmingly on English text. Let's translate the cat sentence into French and German, and the stock-market sentence into Spanish, and see whether "meaning over vocabulary" still holds once vocabulary means an entirely different language.

In [ ]:
# Program 1.2: does mean-pooling hold across languages?

sentences = sentences + [
    "le chat est assis sur le tapis.",         # French: same meaning as sentence 0
    "die Katze saß auf der Matte.",            # German: same meaning as sentence 0
    "El mercado de valores se desplomó ayer.", # Spanish: same meaning as sentence 2
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Look specifically at the German translation: `'The cat sat on the mat.' <-> 'die Katze saß auf der Matte.'` scores *lower* (0.672) than `'The cat sat on the mat.' <-> 'The stock market crashed yesterday.'` (0.746) -- two entirely unrelated English sentences rank as more similar than a sentence and its own faithful German translation.

That's not a bug -- it's a real limit of the technique. `SmolLM2`'s mean-pooling was never trained to align meaning *across* languages; it just averages whatever the model happens to represent, and two English sentences share vocabulary, word order, and token statistics that a simple average leans on heavily, regardless of what they actually mean. French fares a bit better here (Latin script, some shared roots with English), German a bit worse -- but neither is reliable. "Meaning over vocabulary" held within one language; it doesn't automatically survive the trip across languages, and whether it does at all depends entirely on what the underlying model was trained on.

## A real embeddings API

Mean-pooling a small local model's hidden states works, but it's a rough approximation -- `SmolLM2` was never specifically trained to produce good sentence embeddings, in any language. Providers instead offer dedicated **embedding models**, trained precisely for this. Like everywhere else in this course, we can call one through the same OpenAI-compatible client, just changing the endpoint: `embeddings.create` instead of `chat.completions.create`. Let's use Gemini's, on the same sentences.

In [ ]:
# Program 2: sentence embeddings via a real embeddings API (Gemini)

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

def gemini_embedding(text):
    response = gemini.embeddings.create(model="gemini-embedding-001", input=text)
    return torch.tensor(response.data[0].embedding)

gemini_embeddings = [gemini_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(gemini_embeddings[i].unsqueeze(0), gemini_embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Same ranking as our homemade version, but with a clearer gap between the related pair and the unrelated one -- exactly what we'd expect from a model actually trained for this task. Look specifically at the German comparison this time: `'The cat sat on the mat.' <-> 'die Katze saß auf der Matte.'` scores 0.856, clearly ahead of `'The cat sat on the mat.' <-> 'The stock market crashed yesterday.'` at 0.612 -- the anomaly from Program 1.2 is gone. A model trained specifically for embeddings doesn't just do better on average; it fixes exactly the kind of failure we just found.

Note also the vector length: `gemini-embedding-001` returns 3072 numbers per sentence, regardless of how long the sentence is -- a fixed-size summary of its meaning, whether it's fed three words or three paragraphs.

### Your turn: does a real embeddings API handle these two better?

You now have two tools that both claim to turn a sentence into a vector: `sentence_embedding` (Program 1, homemade, mean-pooled) and `gemini_embedding` (Program 2, a model actually trained for this). Let's settle it directly, on two languages from entirely different families -- Chinese and Hungarian (not even Indo-European) -- so that if you happen to read one of them, the other still makes the point honestly.

* Embed each sentence in `mystery_sentences` with `sentence_embedding`, and separately with `gemini_embedding`.
* For each, compute its cosine similarity against every sentence in `sentences`, and find the closest match.
* Do the two tools agree, for both sentences? If not, which one gets it right?

Try it yourself before reading on.

In [ ]:
# Program 3: your turn -- test both embedding tools on two unfamiliar languages

mystery_sentences = [
    "股市昨天崩盤了",              # Chinese
    "A tőzsde tegnap összeomlott.", # Hungarian
]

# your code here

With `sentence_embedding` (Program 1), neither mystery sentence lands correctly: for both the Chinese one and the Hungarian one, the closest match is the French cat sentence -- and the true translation, `'The stock market crashed yesterday.'`, ranks near the very bottom of the six both times (dead last for Hungarian). With `gemini_embedding` (Program 2), both mystery sentences correctly land on the stock-market pair (the English original, or its Spanish twin -- both mean the same thing), by the same wide margin you already saw with German.

Same pattern as before, just more dramatic, and now confirmed on two languages that share almost nothing with English or with each other: `SmolLM2-135M` was trained mostly on English and European-language text, so mean-pooling its hidden states works reasonably within a language family it knows, and breaks down entirely somewhere it barely saw any data -- Chinese and Hungarian, in this case, from two completely unrelated families. `gemini-embedding-001` was trained specifically to place same-meaning sentences close together, in any language, and does exactly that here too.

But `gemini-embedding-001` is a big, general-purpose, hosted model -- every single call leaves your machine, waits on the network, and (Part 2 already showed) eats into a rate-limited free tier. That's fine for three sentences; RAG is going to mean embedding hundreds of chunks, repeatedly. Keep this in mind for Program 4: can a small model, actually built for this one job and run entirely on your own machine, do just as well?

## A local model, actually trained for embeddings

Program 1 already mean-pooled a local model -- the only thing wrong with it was that `SmolLM2` was never *trained* to produce sentence embeddings. Worth being precise about what "trained" means here: neither `SmolLM2` nor what follows is something *we* train. `AutoModel.from_pretrained(name)` downloads an already-trained model's weights straight from the Hugging Face Hub -- "pretrained" means someone else did the training, on data we never see, before we ever call this function. That's been true of every model this course has used since Part 3.

What differs between models is what they were trained *for* -- and a pretrained model can always be specialised further through **fine-tuning**: taking an already-trained model and continuing to train it on a new, more specific objective, instead of starting from scratch. `SmolLM2` stayed generic: trained once, on one objective (predict the next token, so it can generate text), and never fine-tuned for anything else -- mean-pooling its hidden states into a sentence vector is us repurposing a generic model for a job it was never built for. `multilingual-e5-small` took the fine-tuning path: it started life as a generic pretrained multilingual language model, then was further trained specifically to turn a token sequence into a good sentence embedding -- typically by showing it millions of matching pairs (a sentence and its translation, a question and the passage that answers it) and adjusting its weights until mean-pooling naturally pulls each matching pair's vectors close together and pushes unrelated pairs apart. That's exactly the property Program 1.2 found `SmolLM2` never developed.

Let's keep the exact same mean-pooling code, and swap in this fine-tuned model: `multilingual-e5-small` (118M parameters, about the size of the `SmolLM2-135M` we've been using since Part 3 -- so this isn't about a bigger model, it's about one specialised for a different job).

Two details this model expects, which are worth knowing because most embedding models have some equivalent:

* **Prefixes**: stored passages must be prefixed with `passage: ` and questions with `query: `. It was trained that way, and skipping it measurably degrades results.
* **Normalisation**: vectors are scaled to length 1, which makes cosine similarity a plain dot product -- so retrieval later is one matrix multiplication instead of a Python loop.

It's multilingual, which matters here: our documents are in French, our questions often in English -- and, as the exercise just showed, Chinese and Hungarian work too, when the model was actually trained for it.

In [ ]:
# Program 4: a local embedding model, trained for the job -- same three sentences, no API calls

embed_name = "intfloat/multilingual-e5-small"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()  # inference mode again, same reason as Program 1.1

def embed(texts, batch_size=32):
    """Embed a list of texts. Same mean-pooling as Program 1, plus length-1 normalisation,
    and batched so a few hundred passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        # Mean-pool over real tokens only -- padding tokens must not count towards the average.
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

local_embeddings = embed_passages(sentences)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        # Vectors are normalised, so the dot product IS the cosine similarity.
        similarity = (local_embeddings[i] @ local_embeddings[j]).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

# And the mystery sentences, one more time -- still 118M parameters, but trained for exactly this.
for mystery in mystery_sentences:
    mystery_embedding = embed_query(mystery)
    scores = local_embeddings @ mystery_embedding
    print(f"\n{mystery!r} is closest to:")
    print(f"  {sentences[scores.argmax().item()]!r}")

Same ranking again, and both mystery sentences land correctly too -- confirming it really was about training, not size: `multilingual-e5-small` (118M) is smaller than `SmolLM2` (135M) and still gets both right, because it was actually trained on Chinese and Hungarian among dozens of other languages.

## Recap: what actually turns a token into a vector?

Four programs in, "embed this sentence" has been treated a bit like a black box: feed text in, get a vector out. Before moving on, worth pausing on what happens in between -- because it's not quite what Part 4 might have led you to expect, and it's the single most important mechanism behind everything an LLM does.

**Step 1 -- the embedding lookup (Part 4).** Each token id indexes one row of the model's embedding matrix, giving a fixed vector per token -- the *same* vector every single time that token id appears, with zero regard for what surrounds it. "Bank" gets one fixed vector whether the sentence is about a river or a loan. On its own, that's barely more than a fancy lookup table.

**Step 2 -- the secret sauce: self-attention.** `last_hidden_state` -- the thing `sentence_embedding` and `embed` have been mean-pooling this whole notebook -- is *not* that raw lookup. It's what comes out the other end of the model's stack of transformer layers (30 of them, for `SmolLM2-135M`), and each layer runs **self-attention**: every token compares itself against every other token in the sentence and pulls in whatever's relevant, before passing an updated vector to the next layer.

Here's the mechanism, one token at a time, for the sentence "The cat sat down" -- follow along with the diagram below:

1. Every token turns its embedding into three vectors of its own, universally abbreviated **Q**, **K**, **V** (that's what the diagram's boxes are labeled): a **Query** ("what am I looking for?"), a **Key** ("here's what I represent, if anyone's asking"), and a **Value** ("here's what I'll actually contribute"). All three come from the same embedding, through three learned transformations -- nothing outside the sentence is involved.
2. To update "cat"'s vector specifically, take *its* Query and compare it against every token's Key -- including its own. "Compare" means a dot product: a single number per pair, higher when Query and Key point in a similar direction. Say that comes out to 1.0 against "The", 3.0 against "cat" itself, 1.7 against "sat", 1.0 against "down".
3. Those four raw scores get passed through a **softmax**, which turns them into weights that are all positive and sum to exactly 1: roughly 0.09, 0.65, 0.18, 0.09. "Cat" ends up attending mostly to itself, a fair amount to "sat" (the thing it's doing), and only a little to "The" and "down".
4. "Cat"'s new vector is the weighted blend of everyone's *Value* using those weights -- 65% cat's own value, 18% sat's, and a little of the other two. Every other token runs this exact same four-step process in parallel, each with its own Query, so "The", "sat" and "down" also get updated vectors, blended in their own proportions.

Do this at every one of 30 layers, feeding each layer's output into the next as the new starting point, and the vector for a word keeps absorbing more context each time. `SmolLM2` also splits each layer into 9 **attention heads** running in parallel, each with its own Q/K/V transformations -- one head might end up specialising in "which noun does this verb apply to", another in "what does this pronoun refer to", and their outputs are combined before moving to the next layer. Nobody designs the heads to do this; it's just what falls out of training on enough text.

The payoff: "bank" in "I sat by the river bank" ends up attending heavily to "river", while "bank" in "I deposited money at the bank" attends to "deposited" and "money" instead -- same starting embedding, same weights doing the computing, but two different final vectors, because the *content of the sentence* is part of the computation now, not just the identity of the word.

That's the actual secret sauce -- not the embedding matrix from Part 4, but what 30 layers of self-attention do to it. It's exactly why mean-pooling `last_hidden_state` captures more than a plain average of Part 4's static, context-free embeddings ever could -- and why a model trained to make that final representation *specifically* useful for whole-sentence meaning (Programs 2 and 4) does even better than one that wasn't.

<br>
<img src="images/self_attention_academic.jpg" width="550" alt="Self-Attention Diagram" style="display: block; margin: 15px auto;">
<br>

## Key takeaways

* A **sentence embedding** extends token embeddings (Part 4) to a whole sentence -- mean-pool a model's per-token hidden states into one fixed-size vector.
* What actually gets mean-pooled isn't Part 4's static embedding-table lookup -- it's `last_hidden_state`, produced by many layers of self-attention letting every token pull in context from every other token. That's the real secret sauce: the same word ends up with a different vector depending on what surrounds it.
* Mean-pooling only works as well as the underlying model actually understands the input. `SmolLM2` (135M, mostly English/European training data) captures meaning within English, is shakier on European translations, and fails outright on languages from unrelated families (Chinese, Hungarian).
* A model trained *specifically* to produce embeddings -- Gemini's API, or a local model like `multilingual-e5-small` -- fixes this, and it's about training objective, not size: `multilingual-e5-small` (118M) is smaller than `SmolLM2` (135M) and still gets every test right.
* Bulk embedding work belongs on a model you run yourself rather than a hosted API once there's a real corpus to index -- explored properly in Part 8, once there's an actual book and a real, messy web corpus to throw at it.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `AutoModel.from_pretrained(name)` (`transformers`) | a model name | a model exposing raw hidden states, no next-token head | Programs 1.1, 4 |
| `model(**inputs).last_hidden_state` (`transformers`) | tokenized input | one vector per input token | Programs 1.1, 4 |
| `F.normalize(t, dim=-1)` (`torch.nn.functional`) | a tensor | the same vectors scaled to length 1, so a dot product is a cosine | Program 4 |
| `gemini.embeddings.create(model=, input=)` (`openai`) | a model name, a string | an embedding response (`.data[0].embedding`) | Program 2 |